In [20]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F
import torch.utils.data as data


In [21]:
target_discrete_intervals = torch.tensor([100000, 350000]) # 0 to mniej niż 100k$, 1 pomiędzy, 2 więcej
validation_percent = 0.3 # Ile w walidacyjnym

categorical_columns = ["HallwayType", "HeatingType", "AptManageType", "TimeToBusStop", "TimeToSubway", "SubwayStation"]
columns_to_drop = []

In [22]:
def Get_Train_Data():
    train_data = pd.read_csv("train_data.csv")
    train_data = train_data.drop(columns=columns_to_drop)
    return train_data

def Get_Train_Values(train_data):
    train_categorical = pd.get_dummies(train_data[categorical_columns]).astype(int)
    train_numerical = train_data.drop(columns=categorical_columns)
    return train_numerical, train_categorical


In [23]:
def Get_Final_Test_Data():
    final_test_data = pd.read_csv("test_data.csv", index_col=None)
    final_test_data = final_test_data.drop(columns=columns_to_drop)

    return final_test_data

In [24]:
def Get_Final_Test_Values(final_test_data):
    final_test_categorical = pd.get_dummies(final_test_data[categorical_columns]).astype(int)
    final_test_numerical = final_test_data.drop(columns=categorical_columns)
    return final_test_numerical, final_test_categorical

# TODO: 
batch_size=64, shuffle=True

In [25]:
def Get_Train_And_Validate_Loaders(train_numerical, train_categorical, size):
    train_indices = np.random.rand(size)>validation_percent
    print("Train data size: ", size)

    numerical_data = torch.from_numpy(train_numerical.values[train_indices,1:]).float()
    categorical_data = torch.from_numpy(train_categorical.values[train_indices]).float()
    numerical_train_targets = torch.from_numpy(train_numerical.values[train_indices, 0]).float()
    discrete_train_targets = torch.bucketize(numerical_train_targets, target_discrete_intervals) # Dyskretne etykiety

    validate_numerical_data = torch.from_numpy(train_numerical.values[~train_indices, 0:]).float()
    validate_categorical_data = torch.from_numpy(train_categorical.values[~train_indices]).float()
    numerical_validate_targets = torch.from_numpy(train_numerical.values[~train_indices, 0]).float()
    discrete_validate_targets = torch.bucketize(numerical_validate_targets, target_discrete_intervals) # Dyskretne etykiety

    train_loader = data.TensorDataset(numerical_data, categorical_data, discrete_train_targets)
    validate_loader = data.TensorDataset(validate_numerical_data, validate_categorical_data, discrete_validate_targets)

    return train_loader, validate_loader

In [26]:
def Get_Final_Test_Loader(final_test_numerical, final_test_categorical, size):
    print("Final test data size: ", size)

    final_test_numerical_data = torch.from_numpy(final_test_numerical.values).float()
    final_test_categorical_data = torch.from_numpy(final_test_categorical.values).float()

    final_test_loader = data.TensorDataset(final_test_numerical_data, final_test_categorical_data)

    return final_test_loader

In [27]:
def calc_accuracy(pred_targets, targets):
    accuracies = []
    for i in range(3):
        class_correct=(pred_targets == targets.values)[targets == i].sum()
        accuracies.append(class_correct/(targets == i).sum())
    return(np.mean(accuracies))

In [28]:
def Get_Accuracy(model, data_loader):
    targets = []
    preds = []
    model.eval() # <------------------------ trzeba pamiętać żeby dać później model.train()
    for x, cat_x, labels in data_loader:
        x, cat_x, labels = x.to(device), cat_x.to(device), labels.to(device)
        output = model(x, cat_x)
        output = 0
        preds.append(output)
        targets.append(labels)

    accuracy = calc_accuracy(preds, targets)
    return accuracy

In [29]:
def Save_Final_Predictions_To_File(model, final_test_loader):
    model.eval()
    output_array = []
    for x, cat_x in final_test_loader:
        x, cat_x = x.to(device), cat_x.to(device)
        out = model(x, cat_x)
        output_array.append(out)

    pd.Series(output_array).to_csv("pred.csv", index=False, header=False)


In [30]:
train_data = Get_Train_Data()
train_numerical_values, train_categorical_values = Get_Train_Values(train_data)
train_loader, validate_loader = Get_Train_And_Validate_Loaders(train_numerical_values, train_categorical_values, len(train_data))

final_test_data= Get_Final_Test_Data()
final_test_numerical_values, final_test_categorical_values = Get_Final_Test_Values(final_test_data)
final_test_loader = Get_Final_Test_Loader(final_test_numerical_values, final_test_categorical_values, len(final_test_data))

Train data size:  4124
Final test data size:  1767


In [31]:
input_size = train_numerical_values.shape[1] + train_categorical_values.shape[1] - 1
print(input_size)

33


In [37]:
class Price_classifier(nn.Module):
    def __init__(self):
        super(Price_classifier, self).__init__()
        self.layer1 = nn.Linear(input_size, 40)
        self.act_1 = nn.LeakyReLU()
        self.d1 = nn.Dropout(0.4)
        self.layer2 = nn.Linear(40, 20)
        self.act_2 = nn.LeakyReLU()
        self.d2 = nn.Dropout(0.4)
        self.layer3 = nn.Linear(20, 1)
    def forward(self, x, cat_x):
        x = torch.cat([x,cat_x],dim=0) # Tu było dim=1
        activation1 = self.act_1(self.layer1(x))
        activation1 = self.d1(activation1)
        activation2 = self.act_2(self.layer2(activation1))
        activation2 = self.d1(activation2)
        output = self.layer3(activation2)

        return output

In [38]:
# device = torch.device("cuda")
device = torch.device("cpu")
# set_seed(42)
model = Price_classifier().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)

iters = []
losses = []
train_acc = []
val_acc = []
for n in range(1):
    epoch_losses = []
    model.train()
    for x, cat_x, labels in iter(train_loader):
        optimizer.zero_grad()
        x, cat_x, labels = x.to(device), cat_x.to(device), labels.to(device)
        # out= model(x, cat_x)
        out = model(x, cat_x).squeeze()
        loss = criterion(out, labels)
        loss.backward()
        epoch_losses.append(loss.item())
        optimizer.step()

    loss_mean = np.array(epoch_losses).mean()
    iters.append(n)
    losses.append(loss_mean)
    test_acc = Get_Accuracy(model, validate_loader)
    print(f"Epoch {n} loss {loss_mean:.3} test_acc: {test_acc:.3}")
    train_acc.append(Get_Accuracy(model, train_loader)) # compute training accuracy
    val_acc.append(test_acc)  # compute validation accuracy


print("Final Training Accuracy: {}".format(train_acc[-1]))
print("Final Validation Accuracy: {}".format(val_acc[-1]))

RuntimeError: result type Float can't be cast to the desired output type Long